In [1]:
import os
import re
import io
import ipywidgets as widgets
from IPython.display import display
from pypdf import PdfReader
from nltk.corpus import stopwords
import pickle

def clean_text(text):
    """
    Clean text by removing unwanted characters and formatting.
    """
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # Remove non-ASCII characters
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove special characters and numbers
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespace
    text = re.sub(r'\n+', '\n', text)  # Remove extra newlines
    return text


def remove_stop_words(text):
    stop_words = set(stopwords.words('english'))
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)


def preprocess_pdf(file_content, filename):
    """Extract and preprocess text from uploaded PDF file."""
    try:
        # Convert memoryview to BytesIO
        pdf_stream = io.BytesIO(file_content)

        reader = PdfReader(pdf_stream)
        extracted_text = ""
        for page in reader.pages:
            text = page.extract_text()
            if text:
                extracted_text += text + "\n"

        cleaned_text = clean_text(extracted_text)
        cleaned_text = remove_stop_words(cleaned_text)

        # Save cleaned text to file
        output_path = os.path.join(os.getcwd(), f"{filename}.txt")
        with open(output_path, "w", encoding="utf-8") as text_file:
            text_file.write(cleaned_text)

        print(f"Processed text saved to: {output_path}")
        print(f"Preview:\n{cleaned_text[:500]}...")  # Print first 500 chars

        # Save metadata for second script
        metadata = {
            "filename": filename,
            "file_path": output_path
        }
        with open("processed_file.pkl", "wb") as f:
            pickle.dump(metadata, f)

        print("Metadata saved for evaluation script.")

    except Exception as e:
        print(f"Error processing PDF: {e}")


# Upload Widget
upload_widget = widgets.FileUpload(
    accept='.pdf',  # Accept only PDF files
    multiple=False  # Only allow single file upload
)


def on_upload_change(change):
    """Handle file upload and process PDF."""
    if upload_widget.value:
        for file_info in upload_widget.value:
            filename = file_info['name'].replace(".pdf", "")
            file_content = file_info['content'].tobytes()  # Convert memoryview to bytes
            
            # Process PDF
            preprocess_pdf(file_content, filename)


# Attach event listener
upload_widget.observe(on_upload_change, names='value')

display(upload_widget)

FileUpload(value=(), accept='.pdf', description='Upload')

In [8]:
for file_info in upload_widget.value:
    print(file_info.name)

pillar3-disclosures-1q-2018.pdf


In [2]:
print(upload_widget.value)

({'name': 'pillar3-disclosures-1q-2018.pdf', 'type': 'application/pdf', 'size': 98305, 'content': <memory at 0x15230fac0>, 'last_modified': datetime.datetime(2025, 3, 1, 7, 48, 13, 135000, tzinfo=datetime.timezone.utc)},)


In [37]:
import os
import pickle
import json
import pandas as pd
import boto3
from botocore.config import Config
from dotenv import load_dotenv

load_dotenv("codes.env")

# AWS credentials
aws_access_key = os.environ.get("AWS_ACCESS_KEY_ID")
aws_secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY")
aws_region = os.environ.get("AWS_REGION")

# AWS Bedrock model configuration
MODEL_ID_LLAMA = "arn:aws:bedrock:us-west-2:874280117166:inference-profile/us.meta.llama3-3-70b-instruct-v1:0"

# Prevent Bedrock timeout
config = Config(read_timeout=1000)

client = boto3.client(
    "bedrock-runtime",
    region_name=aws_region,
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    config=config
)

# Load topic mappings
mapping_file_path = 'final_file_topic_mapping.csv'
file_topic_mapping = pd.read_csv(mapping_file_path)
unique_topics = file_topic_mapping['folder_name'].unique().tolist()
unique_topics_str = ', '.join(unique_topics)


def read_txt_file(file_path):
    """Reads the content of a .txt file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            return file.read()
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return None


def map_to_category(predicted_output):
    """Maps the model's output to a known category."""
    predicted_output = predicted_output.lower().strip()
    for topic in unique_topics:
        if topic.lower() in predicted_output:
            return topic
    return "unknown"


def evaluate_topic_with_llama(file_content):
    """Classify the text using AWS Bedrock (Meta's Llama 3.3 70B Instruct)."""
    try:
        prompt = f"Classify the following text into only one of these topics: {unique_topics_str}. \n{file_content}"
        formatted_prompt = f"""
            <|begin_of_text|>
            <|start_header_id|>user<|end_header_id|>
            {prompt}
            <|eot_id|>
            <|start_header_id|>assistant<|end_header_id|>
            """

        response = client.invoke_model(
            modelId=MODEL_ID_LLAMA,
            body=json.dumps({
                "prompt": formatted_prompt,
                "max_gen_len": 512,
                "temperature": 0,
            }),
            contentType="application/json"
        )
        response_body = json.loads(response['body'].read())
        predicted_topic = response_body.get("generation", "").strip()
        
        if not predicted_topic:
            print("Empty response from AWS Bedrock Llama, defaulting to unknown.")

        return map_to_category(predicted_topic)

    except Exception as e:
        print(f"Error calling AWS Bedrock API: {e}")
        return "unknown"


def evaluate_saved_file():
    """Loads metadata, reads file content, and evaluates it."""
    try:
        # Load metadata
        with open("processed_file.pkl", "rb") as f:
            metadata = pickle.load(f)

        filename = metadata["filename"]
        file_path = metadata["file_path"]

        print(f"Evaluating file: {filename}")

        # Read file content
        text_content = read_txt_file(file_path)
        if text_content:
            predicted_topic = evaluate_topic_with_llama(text_content)
            print(f"Predicted Topic: {predicted_topic}")
        else:
            print("Error: No content found in the file.")

    except FileNotFoundError:
        print("Error: No processed file metadata found. Run `upload_pdf.py` first.")
    except Exception as e:
        print(f"Unexpected error: {e}")


# Run the evaluation
evaluate_saved_file()

Evaluating file: 626 Banks_GCO vetted
Predicted Topic: Anti Money Laundering


# showing demo of explanation before predicted topic:

In [3]:
import os
import pickle
import json
import pandas as pd
import boto3
import re
from botocore.config import Config
from dotenv import load_dotenv

load_dotenv("codes.env")

# AWS credentials
aws_access_key = os.environ.get("AWS_ACCESS_KEY_ID")
aws_secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY")
aws_region = os.environ.get("AWS_REGION")

# AWS Bedrock model configuration
MODEL_ID_LLAMA = "us.meta.llama3-3-70b-instruct-v1:0"

# Prevent Bedrock timeout
config = Config(read_timeout=1000)

client = boto3.client(
    "bedrock-runtime",
    region_name=aws_region,
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    config=config
)

# Load topic mappings
mapping_file_path = 'final_file_topic_mapping.csv'
file_topic_mapping = pd.read_csv(mapping_file_path)
unique_topics = file_topic_mapping['folder_name'].unique().tolist()
unique_topics_str = ', '.join(unique_topics)

def read_txt_file(file_path):
    """Reads the content of a .txt file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            return file.read()
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return None

def extract_final_topic(response_text):
    """
    Extracts the final topic from the model's response using regex.
    Ensures that we capture the topic stated explicitly at the end.
    """
    match = re.search(r"Final Topic:\s*(.+)", response_text, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    
    # If no clear label is found, fall back to the last line
    lines = response_text.strip().split("\n")
    return lines[-1].strip() if lines else "unknown"

def evaluate_topic_with_llama(file_content):
    """Classify the text using AWS Bedrock (Meta's Llama 3.3 70B Instruct)."""
    try:
        prompt = f"Classify the following text into only one of these topics: {unique_topics_str}. \n{file_content}"
        formatted_prompt = f"""
            <|begin_of_text|>
            <|start_header_id|>user<|end_header_id|>
            {prompt}
            <|eot_id|>
            <|start_header_id|>assistant<|end_header_id|>
            """

        response = client.invoke_model(
            modelId=MODEL_ID_LLAMA,
            body=json.dumps({
                "prompt": formatted_prompt,
                "max_gen_len": 512,
                "temperature": 0,
            }),
            contentType="application/json"
        )
        response_body = json.loads(response['body'].read())
        predicted_topic = response_body.get("generation", "").strip()
        
        if not predicted_topic:
            print("Empty response from AWS Bedrock Llama, defaulting to unknown.")

        return map_to_category(predicted_topic)

    except Exception as e:
        print(f"Error calling AWS Bedrock API: {e}")
        return "unknown"

def evaluate_saved_file():
    """Loads metadata, reads file content, and evaluates it with explanation."""
    try:
        # Load metadata
        with open("processed_file.pkl", "rb") as f:
            metadata = pickle.load(f)

        filename = metadata["filename"]
        file_path = metadata["file_path"]

        print(f"Evaluating file: {filename}")

        # Read file content
        text_content = read_txt_file(file_path)
        if text_content:
            explanation, predicted_topic = evaluate_topic_with_llama(text_content)
            print(f"Explanation: {explanation}\nPredicted Topic: {predicted_topic}")
        else:
            print("Error: No content found in the file.")

    except FileNotFoundError:
        print("Error: No processed file metadata found. Run `upload_pdf.py` first.")
    except Exception as e:
        print(f"Unexpected error: {e}")

# Run the evaluation

evaluate_saved_file()

Evaluating file: pillar3-disclosures-1q-2018
Error calling AWS Bedrock API: name 'map_to_category' is not defined
Unexpected error: too many values to unpack (expected 2)


# Demo with Rule Based Classification

In [ ]:
import os
import re
import io
import ipywidgets as widgets
from IPython.display import display
from pypdf import PdfReader
from nltk.corpus import stopwords
import pickle

def clean_text(text):
    """
    Clean text by removing unwanted characters and formatting.
    """
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # Remove non-ASCII characters
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove special characters and numbers
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespace
    text = re.sub(r'\n+', '\n', text)  # Remove extra newlines
    return text


def remove_stop_words(text):
    stop_words = set(stopwords.words('english'))
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)


def preprocess_pdf(file_content, filename):
    """Extract and preprocess text from uploaded PDF file."""
    try:
        # Convert memoryview to BytesIO
        pdf_stream = io.BytesIO(file_content)

        reader = PdfReader(pdf_stream)
        extracted_text = ""
        for page in reader.pages:
            text = page.extract_text()
            if text:
                extracted_text += text + "\n"

        cleaned_text = clean_text(extracted_text)
        cleaned_text = remove_stop_words(cleaned_text)

        # Save cleaned text to file
        output_path = os.path.join(os.getcwd(), f"{filename}.txt")
        with open(output_path, "w", encoding="utf-8") as text_file:
            text_file.write(cleaned_text)

        print(f"Processed text saved to: {output_path}")
        print(f"Preview:\n{cleaned_text[:500]}...")  # Print first 500 chars

        # Save metadata for second script
        metadata = {
            "filename": filename,
            "file_path": output_path
        }
        with open("processed_file.pkl", "wb") as f:
            pickle.dump(metadata, f)

        print("Metadata saved for evaluation script.")

    except Exception as e:
        print(f"Error processing PDF: {e}")


# Upload Widget
upload_widget = widgets.FileUpload(
    accept='.pdf',  # Accept only PDF files
    multiple=False  # Only allow single file upload
)


def on_upload_change(change):
    """Handle file upload and process PDF."""
    if upload_widget.value:
        for file_info in upload_widget.value:
            filename = file_info['name'].replace(".pdf", "")
            file_content = file_info['content'].tobytes()  # Convert memoryview to bytes
            
            # Process PDF
            preprocess_pdf(file_content, filename)


# Attach event listener
upload_widget.observe(on_upload_change, names='value')

display(upload_widget)

In [13]:
print(upload_widget.value)

({'name': '626 Banks_GCO vetted.pdf', 'type': 'application/pdf', 'size': 166662, 'content': <memory at 0x17a5ca680>, 'last_modified': datetime.datetime(2025, 3, 1, 7, 10, 57, 155000, tzinfo=datetime.timezone.utc)},)


In [20]:
import os
import pickle
import json
import pandas as pd
import math
import re
from sklearn.feature_extraction.text import CountVectorizer
import boto3
from botocore.config import Config
from dotenv import load_dotenv

#  Load Environment Variables
load_dotenv("codes.env")

aws_access_key = os.environ.get("AWS_ACCESS_KEY_ID")
aws_secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY")
aws_region = os.environ.get("AWS_REGION")

MODEL_ID_LLAMA = "us.meta.llama3-3-70b-instruct-v1:0"
config = Config(read_timeout=1000)

bedrock_client = boto3.client(
    "bedrock-runtime",
    region_name=aws_region,
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    config=config
)

#  Load Predefined Topics & Mappings
mapping_file_path = 'final_file_topic_mapping.csv'
file_topic_mapping = pd.read_csv(mapping_file_path)
unique_topics = file_topic_mapping['folder_name'].unique().tolist()
unique_topics_str = ', '.join(unique_topics)

#  Load Keyword Data for Rule-Based Classification
top_keywords_per_topic = {}
main_topics = set()

df = pd.read_csv("refined_tfidf_bigrams.csv")
for index, row in df.iterrows():
    topic_name = row.iloc[0].strip()
    keywords = [row[col] for col in df.columns[1:] if pd.notna(row[col])]
    if keywords:
        top_keywords_per_topic[topic_name] = keywords[:150]
        main_topics.add(topic_name.split("/")[0])

#  Tokenization (For Rule-Based Classifier)
vectorizer = CountVectorizer(
    stop_words="english",
    lowercase=True,
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=(1, 2)
)

def fast_tokenize(text):
    return set(vectorizer.build_analyzer()(text))

#  Length Penalty
def apply_length_penalty(score, doc_length):
    return score / (1 + math.log(1 + doc_length) / 70)

#  Rule-Based Classification
def classify_document(text):
    doc_words = fast_tokenize(text)
    doc_length = len(doc_words)

    if doc_length < 100:
        return None, 0.0  # Skip classification if document too short

    best_match, best_score = None, 0
    for topic in main_topics:
        topic_keywords = set(word for subtopic in top_keywords_per_topic if subtopic.startswith(topic) for word in top_keywords_per_topic[subtopic])
        matched_words = doc_words.intersection(topic_keywords)

        weighted_score = sum(1.0 * (0.85 ** idx) for idx, word in enumerate(matched_words))
        max_possible_score = sum(1.0 * (0.85 ** idx) for idx in range(len(topic_keywords)))
        normalized_score = weighted_score / max_possible_score if max_possible_score > 0 else 0

        adjusted_score = apply_length_penalty(normalized_score, doc_length)

        if adjusted_score > best_score and len(matched_words) >= 30:
            best_score = adjusted_score
            best_match = topic

    if best_score < 0.9:
        return None, best_score  

    return best_match, best_score

#  LLM
def evaluate_topic_with_llama(file_content):
    """Classify the text using AWS Bedrock (Meta's Llama 3.3 70B Instruct)."""
    try:
        prompt = f"""
        Analyze the following document and classify it into only one of these topics: {unique_topics_str}.
        After explaining your reasoning, clearly state the final topic at the end.

        Document:
        {file_content}

        Explanation:
        Final Topic:
        """

        formatted_prompt = f"""
        <|begin_of_text|>
        <|start_header_id|>user<|end_header_id|>
        {prompt}
        <|eot_id|>
        <|start_header_id|>assistant<|end_header_id|>
        """

        response = bedrock_client.invoke_model(
            modelId=MODEL_ID_LLAMA,
            body=json.dumps({
                "prompt": formatted_prompt,
                "max_gen_len": 512,
                "temperature": 0,
            }),
            contentType="application/json"
        )

        response_body = json.loads(response['body'].read())
        response_text = response_body.get("generation", "").strip()

        match = re.search(r"Final Topic:\s*(.+)", response_text, re.IGNORECASE)
        predicted_topic = match.group(1).strip() if match else "Unknown"

        return predicted_topic, response_text

    except Exception as e:
        print(f"Error calling AWS Bedrock API: {e}")
        return "Unknown", "Error occurred during LLM call"

#  Main Pipeline (Single File Upload)
def process_single_file():
    with open("processed_file.pkl", "rb") as f:
        metadata = pickle.load(f)

    file_path = metadata["file_path"]
    filename = metadata["filename"]

    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Rule-Based Attempt
    predicted_topic, confidence = classify_document(content)

    if predicted_topic:
        print(f" Rule-Based Classification: {predicted_topic} (Confidence: {confidence:.2f})")
        return filename, predicted_topic, confidence, "-", "Rule-Based"

    # Fallback to LLM (full document)
    predicted_topic, explanation = evaluate_topic_with_llama(content)

    print(f"LLM Classification: {predicted_topic}")
    return filename, predicted_topic, "-", explanation, "LLM"

#  Run and Output Results
if __name__ == "__main__":
    filename, predicted_topic, confidence, explanation, source = process_single_file()

    result = {
        "Filename": filename,
        "Predicted Topic": predicted_topic,
        "Confidence": confidence,
        "Explanation": explanation,
        "Source": source
    }
    print(json.dumps(result, indent=4))


LLM Classification: Annual Reports
{
    "Filename": "gp-financial-1q-2018",
    "Predicted Topic": "Annual Reports",
    "Confidence": "-",
    "Explanation": "The document provided appears to be a financial report of United Overseas Bank Limited, detailing its unaudited financial results for the first quarter ended March. The report includes information on the bank's financial performance, such as net interest income, non-interest income, operating expenses, and allowance for expected credit losses. It also provides an analysis of the bank's business segments, geographical segments, and capital adequacy ratios.\n\nThe report is written in a formal and technical tone, using financial terminology and jargon, which suggests that it is intended for an audience with a background in finance or banking. The level of detail and the specific information provided, such as the bank's financial statements and capital adequacy ratios, also suggest that the report is intended for regulatory or inv

# Demo with Rule Based Classification and Sample

In [21]:
import os
import re
import io
import ipywidgets as widgets
from IPython.display import display
from pypdf import PdfReader
from nltk.corpus import stopwords
import pickle

def clean_text(text):
    """
    Clean text by removing unwanted characters and formatting.
    """
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # Remove non-ASCII characters
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove special characters and numbers
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespace
    text = re.sub(r'\n+', '\n', text)  # Remove extra newlines
    return text


def remove_stop_words(text):
    stop_words = set(stopwords.words('english'))
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)


def preprocess_pdf(file_content, filename):
    """Extract and preprocess text from uploaded PDF file."""
    try:
        # Convert memoryview to BytesIO
        pdf_stream = io.BytesIO(file_content)

        reader = PdfReader(pdf_stream)
        extracted_text = ""
        for page in reader.pages:
            text = page.extract_text()
            if text:
                extracted_text += text + "\n"

        cleaned_text = clean_text(extracted_text)
        cleaned_text = remove_stop_words(cleaned_text)

        # Save cleaned text to file
        output_path = os.path.join(os.getcwd(), f"{filename}.txt")
        with open(output_path, "w", encoding="utf-8") as text_file:
            text_file.write(cleaned_text)

        print(f"Processed text saved to: {output_path}")
        print(f"Preview:\n{cleaned_text[:500]}...")  # Print first 500 chars

        # Save metadata for second script
        metadata = {
            "filename": filename,
            "file_path": output_path
        }
        with open("processed_file.pkl", "wb") as f:
            pickle.dump(metadata, f)

        print("Metadata saved for evaluation script.")

    except Exception as e:
        print(f"Error processing PDF: {e}")


# Upload Widget
upload_widget = widgets.FileUpload(
    accept='.pdf',  # Accept only PDF files
    multiple=False  # Only allow single file upload
)


def on_upload_change(change):
    """Handle file upload and process PDF."""
    if upload_widget.value:
        for file_info in upload_widget.value:
            filename = file_info['name'].replace(".pdf", "")
            file_content = file_info['content'].tobytes()  # Convert memoryview to bytes
            
            # Process PDF
            preprocess_pdf(file_content, filename)


# Attach event listener
upload_widget.observe(on_upload_change, names='value')

display(upload_widget)

FileUpload(value=(), accept='.pdf', description='Upload')

In [22]:
import os
import pickle
import json
import random
import pandas as pd
import math
import re
from sklearn.feature_extraction.text import CountVectorizer
import boto3
from botocore.config import Config
from dotenv import load_dotenv

#  Load Environment Variables
load_dotenv("codes.env")

aws_access_key = os.environ.get("AWS_ACCESS_KEY_ID")
aws_secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY")
aws_region = os.environ.get("AWS_REGION")

MODEL_ID_LLAMA = "us.meta.llama3-3-70b-instruct-v1:0"
config = Config(read_timeout=1000)

bedrock_client = boto3.client(
    "bedrock-runtime",
    region_name=aws_region,
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    config=config
)

#  Load Predefined Topics & Mappings
mapping_file_path = 'final_file_topic_mapping.csv'
file_topic_mapping = pd.read_csv(mapping_file_path)
unique_topics = file_topic_mapping['folder_name'].unique().tolist()
unique_topics_str = ', '.join(unique_topics)

#  Load Keyword Data for Rule-Based Classification
top_keywords_per_topic = {}
main_topics = set()

df = pd.read_csv("refined_tfidf_bigrams.csv")
for index, row in df.iterrows():
    topic_name = row.iloc[0].strip()
    keywords = [row[col] for col in df.columns[1:] if pd.notna(row[col])]
    if keywords:
        top_keywords_per_topic[topic_name] = keywords[:150]
        main_topics.add(topic_name.split("/")[0])

#  Tokenization (For Rule-Based Classifier)
vectorizer = CountVectorizer(
    stop_words="english",
    lowercase=True,
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=(1, 2)
)

def fast_tokenize(text):
    return set(vectorizer.build_analyzer()(text))

#  Length Penalty
def apply_length_penalty(score, doc_length):
    return score / (1 + math.log(1 + doc_length) / 70)

#  Rule-Based Classification
def classify_document(text):
    doc_words = fast_tokenize(text)
    doc_length = len(doc_words)

    if doc_length < 100:
        return None, 0.0  # Skip classification if document too short

    best_match, best_score = None, 0
    for topic in main_topics:
        topic_keywords = set(word for subtopic in top_keywords_per_topic if subtopic.startswith(topic) for word in top_keywords_per_topic[subtopic])
        matched_words = doc_words.intersection(topic_keywords)

        weighted_score = sum(1.0 * (0.85 ** idx) for idx, word in enumerate(matched_words))
        max_possible_score = sum(1.0 * (0.85 ** idx) for idx in range(len(topic_keywords)))
        normalized_score = weighted_score / max_possible_score if max_possible_score > 0 else 0

        adjusted_score = apply_length_penalty(normalized_score, doc_length)

        if adjusted_score > best_score and len(matched_words) >= 30:
            best_score = adjusted_score
            best_match = topic

    if best_score < 0.9:
        return None, best_score  # Low confidence → No classification (fall back to LLM)

    return best_match, best_score

#  Sampling Logic for Hybrid Text Extraction (Intro, Middle Sample, Conclusion)
def extract_intro_middle_conclusion(text, max_tokens=20000):
    words = text.split()
    total_words = len(words)

    if total_words < 5000:
        ratios = (0.15, 0.15, 0.15)
    elif total_words < 20000:
        ratios = (0.08, 0.12, 0.08)
    elif total_words < 50000:
        ratios = (0.04, 0.08, 0.04)
    else:
        ratios = (0.02, 0.06, 0.02)

    intro_end = max(int(total_words * ratios[0]), 100)
    conclusion_start = max(int(total_words * (1 - ratios[2])), total_words - 100)

    middle = words[intro_end:conclusion_start]
    middle_sample_size = int(len(middle) * ratios[1])
    middle_sample = random.sample(middle, min(middle_sample_size, len(middle)))

    hybrid_text_words = words[:intro_end] + middle_sample + words[conclusion_start:]
    estimated_tokens = len(hybrid_text_words) * 1.3

    if estimated_tokens > max_tokens:
        allowed_words = int(max_tokens / 1.3)
        hybrid_text_words = hybrid_text_words[:allowed_words]

    return " ".join(hybrid_text_words)

#  LLM Fallback (Uses Sampled Text)
def evaluate_topic_with_llama(file_content):
    try:
        prompt = f"""
        Analyze the following document sample and classify it into only one of these topics: {unique_topics_str}.
        After explaining your reasoning, clearly state the final topic at the end.

        Document:
        {file_content}

        Explanation:
        Final Topic:
        """

        formatted_prompt = f"""
        <|begin_of_text|>
        <|start_header_id|>user<|end_header_id|>
        {prompt}
        <|eot_id|>
        <|start_header_id|>assistant<|end_header_id|>
        """

        response = bedrock_client.invoke_model(
            modelId=MODEL_ID_LLAMA,
            body=json.dumps({
                "prompt": formatted_prompt,
                "max_gen_len": 512,
                "temperature": 0,
            }),
            contentType="application/json"
        )

        response_body = json.loads(response['body'].read())
        response_text = response_body.get("generation", "").strip()

        match = re.search(r"Final Topic:\s*(.+)", response_text, re.IGNORECASE)
        predicted_topic = match.group(1).strip() if match else "Unknown"

        return predicted_topic, response_text

    except Exception as e:
        print(f"Error calling AWS Bedrock API: {e}")
        return "Unknown", "Error occurred during LLM call"

#  Main Pipeline (Single File Upload)
def process_single_file():
    with open("processed_file.pkl", "rb") as f:
        metadata = pickle.load(f)

    file_path = metadata["file_path"]
    filename = metadata["filename"]

    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Rule-Based Attempt
    predicted_topic, confidence = classify_document(content)

    if predicted_topic:
        print(f" Rule-Based Classification: {predicted_topic} (Confidence: {confidence:.2f})")
        return filename, predicted_topic, confidence, "-", "Rule-Based"

    # Fallback to LLM using Sampled Text (Hybrid)
    sampled_text = extract_intro_middle_conclusion(content)
    predicted_topic, explanation = evaluate_topic_with_llama(sampled_text)

    print(f" LLM Classification: {predicted_topic}")
    return filename, predicted_topic, "-", explanation, "LLM"

#  Run and Output Results
if __name__ == "__main__":
    filename, predicted_topic, confidence, explanation, source = process_single_file()

    result = {
        "Filename": filename,
        "Predicted Topic": predicted_topic,
        "Confidence": confidence,
        "Explanation": explanation,
        "Source": source
    }
    print(json.dumps(result, indent=4))


 Rule-Based Classification: Annual Reports (Confidence: 0.90)
{
    "Filename": "selected-financial-statements-2021-en",
    "Predicted Topic": "Annual Reports",
    "Confidence": 0.9012374092614694,
    "Explanation": "-",
    "Source": "Rule-Based"
}


# Replaced Rule-Based Classification with Trainable Random Forest Model

In [5]:
import os
import re
import io
import ipywidgets as widgets
from IPython.display import display
from pypdf import PdfReader
from nltk.corpus import stopwords
import pickle

def clean_text(text):
    """
    Clean text by removing unwanted characters and formatting.
    """
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # Remove non-ASCII characters
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove special characters and numbers
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespace
    text = re.sub(r'\n+', '\n', text)  # Remove extra newlines
    return text


def remove_stop_words(text):
    stop_words = set(stopwords.words('english'))
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)


def preprocess_pdf(file_content, filename):
    """Extract and preprocess text from uploaded PDF file."""
    try:
        # Convert memoryview to BytesIO
        pdf_stream = io.BytesIO(file_content)

        reader = PdfReader(pdf_stream)
        extracted_text = ""
        for page in reader.pages:
            text = page.extract_text()
            if text:
                extracted_text += text + "\n"

        cleaned_text = clean_text(extracted_text)
        cleaned_text = remove_stop_words(cleaned_text)

        # Save cleaned text to file
        output_path = os.path.join(os.getcwd(), f"{filename}.txt")
        with open(output_path, "w", encoding="utf-8") as text_file:
            text_file.write(cleaned_text)

        print(f"Processed text saved to: {output_path}")
        print(f"Preview:\n{cleaned_text[:500]}...")  # Print first 500 chars

        # Save metadata for second script
        metadata = {
            "filename": filename,
            "file_path": output_path
        }
        with open("processed_file.pkl", "wb") as f:
            pickle.dump(metadata, f)

        print("Metadata saved for evaluation script.")

    except Exception as e:
        print(f"Error processing PDF: {e}")


# Upload Widget
upload_widget = widgets.FileUpload(
    accept='.pdf',  # Accept only PDF files
    multiple=False  # Only allow single file upload
)


def on_upload_change(change):
    """Handle file upload and process PDF."""
    if upload_widget.value:
        for file_info in upload_widget.value:
            filename = file_info['name'].replace(".pdf", "")
            file_content = file_info['content'].tobytes()  # Convert memoryview to bytes
            
            # Process PDF
            preprocess_pdf(file_content, filename)


# Attach event listener
upload_widget.observe(on_upload_change, names='value')

display(upload_widget)

FileUpload(value=(), accept='.pdf', description='Upload')

In [ ]:
import os
import pickle
import json
import random
import pandas as pd
import math
import re
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
import boto3
from botocore.config import Config
from dotenv import load_dotenv

#  Load Environment Variables
load_dotenv("codes.env")

aws_access_key = os.environ.get("AWS_ACCESS_KEY_ID")
aws_secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY")
aws_region = os.environ.get("AWS_REGION")

MODEL_ID_LLAMA = "us.meta.llama3-3-70b-instruct-v1:0"
config = Config(read_timeout=1000)

bedrock_client = boto3.client(
    "bedrock-runtime",
    region_name=aws_region,
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    config=config
)

#  Load Predefined Topics & Mappings
mapping_file_path = 'final_file_topic_mapping.csv'
file_topic_mapping = pd.read_csv(mapping_file_path)
unique_topics = file_topic_mapping['folder_name'].unique().tolist()
unique_topics_str = ', '.join(unique_topics)

#  Load Trained Model & Vectorizer** 
print(" Loading Trained TF-IDF Vectorizer and Random Forest Model...")
tfidf_vectorizer = joblib.load("tfidf_vectorizer.pkl")
rf_model = joblib.load("rf_model.pkl")

#  Random Forest Classification** 
def rf_classify_document(text, confidence_threshold=0.99):
    """Classifies a document using the trained Random Forest model."""
    text_tfidf = tfidf_vectorizer.transform([text])  # Convert text to TF-IDF features
    y_pred_proba = rf_model.predict_proba(text_tfidf)[0]  # Get probability scores
    max_prob = max(y_pred_proba)
    
    if max_prob < confidence_threshold:
        return None, max_prob  # Defer to LLM

    predicted_topic = rf_model.classes_[y_pred_proba.argmax()]
    return predicted_topic, max_prob

#  Sampling Logic for Hybrid Text Extraction**
def extract_intro_middle_conclusion(text, max_tokens=20000):
    words = text.split()
    total_words = len(words)

    if total_words < 5000:
        ratios = (0.15, 0.15, 0.15)
    elif total_words < 20000:
        ratios = (0.08, 0.12, 0.08)
    elif total_words < 50000:
        ratios = (0.04, 0.08, 0.04)
    else:
        ratios = (0.02, 0.06, 0.02)

    intro_end = max(int(total_words * ratios[0]), 100)
    conclusion_start = max(int(total_words * (1 - ratios[2])), total_words - 100)

    middle = words[intro_end:conclusion_start]
    middle_sample_size = int(len(middle) * ratios[1])
    middle_sample = random.sample(middle, min(middle_sample_size, len(middle)))

    hybrid_text_words = words[:intro_end] + middle_sample + words[conclusion_start:]
    estimated_tokens = len(hybrid_text_words) * 1.3

    if estimated_tokens > max_tokens:
        allowed_words = int(max_tokens / 1.3)
        hybrid_text_words = hybrid_text_words[:allowed_words]

    return " ".join(hybrid_text_words)

#  LLM Fallback (Uses Sampled Text)**
def evaluate_topic_with_llama(file_content):
    try:
        prompt = f"""
        Analyze the following document sample and classify it into only one of these topics: {unique_topics_str}.
        After explaining your reasoning, clearly state the final topic at the end.

        Document:
        {file_content}

        Explanation:
        Final Topic:
        """

        formatted_prompt = f"""
        <|begin_of_text|>
        <|start_header_id|>user<|end_header_id|>
        {prompt}
        <|eot_id|>
        <|start_header_id|>assistant<|end_header_id|>
        """

        response = bedrock_client.invoke_model(
            modelId=MODEL_ID_LLAMA,
            body=json.dumps({
                "prompt": formatted_prompt,
                "max_gen_len": 512,
                "temperature": 0,
            }),
            contentType="application/json"
        )

        response_body = json.loads(response['body'].read())
        response_text = response_body.get("generation", "").strip()

        match = re.search(r"Final Topic:\s*(.+)", response_text, re.IGNORECASE)
        predicted_topic = match.group(1).strip() if match else "Unknown"

        return predicted_topic, response_text

    except Exception as e:
        print(f"Error calling AWS Bedrock API: {e}")
        return "Unknown", "Error occurred during LLM call"

#  Main Pipeline (Single File Upload)**
def process_single_file():
    with open("processed_file.pkl", "rb") as f:
        metadata = pickle.load(f)

    file_path = metadata["file_path"]
    filename = metadata["filename"]

    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    sampled_text = extract_intro_middle_conclusion(content)
    predicted_topic, confidence = rf_classify_document(sampled_text)

    if predicted_topic:
        print(f" Random Forest Classification: {predicted_topic} (Confidence: {confidence:.2f})")
        return filename, predicted_topic, confidence, "-", "Random Forest Rule-Based Classification"

    # Fallback to LLM using Sampled Text (Hybrid)
    predicted_topic, explanation = evaluate_topic_with_llama(sampled_text)

    print(f" LLM Classification: {predicted_topic}")
    return filename, predicted_topic, "-", explanation, "LLM"

#  Run and Output Results**
if __name__ == "__main__":
    filename, predicted_topic, confidence, explanation, source = process_single_file()

    result = {
        "Filename": filename,
        "Predicted Topic": predicted_topic,
        "Confidence": confidence,
        "Explanation": explanation,
        "Source": source
    }
    print(json.dumps(result, indent=4))


 Loading Trained TF-IDF Vectorizer and Random Forest Model...
✅ Max Prob: 1.0000, Predicted Topic: Risk Management
✅ Top 5 Probabilities: [1.0, 0.0, 0.0, 0.0, 0.0]
 Random Forest Classification: Risk Management (Confidence: 1.00)
{
    "Filename": "pillar3-disclosures-2q-2017",
    "Predicted Topic": "Risk Management",
    "Confidence": 1.0,
    "Explanation": "-",
    "Source": "Random Forest Rule-Based Classification"
}


# Final Rule-Based Classification with Trainable Random Forest
Under this section, we defer all topics predicted as "Financial Regulations" to the Large Language Model, since precision for Financial Regulations is significantly lower than the rest.

In [1]:
import os
import re
import io
import ipywidgets as widgets
from IPython.display import display
from pypdf import PdfReader
from nltk.corpus import stopwords
import pickle

def clean_text(text):
    """
    Clean text by removing unwanted characters and formatting.
    """
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # Remove non-ASCII characters
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove special characters and numbers
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespace
    text = re.sub(r'\n+', '\n', text)  # Remove extra newlines
    return text


def remove_stop_words(text):
    stop_words = set(stopwords.words('english'))
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)


def preprocess_pdf(file_content, filename):
    """Extract and preprocess text from uploaded PDF file."""
    try:
        # Convert memoryview to BytesIO
        pdf_stream = io.BytesIO(file_content)

        reader = PdfReader(pdf_stream)
        extracted_text = ""
        for page in reader.pages:
            text = page.extract_text()
            if text:
                extracted_text += text + "\n"

        cleaned_text = clean_text(extracted_text)
        cleaned_text = remove_stop_words(cleaned_text)

        # Save cleaned text to file
        output_path = os.path.join(os.getcwd(), f"{filename}.txt")
        with open(output_path, "w", encoding="utf-8") as text_file:
            text_file.write(cleaned_text)

        print(f"Processed text saved to: {output_path}")
        print(f"Preview:\n{cleaned_text[:500]}...")  # Print first 500 chars

        # Save metadata for second script
        metadata = {
            "filename": filename,
            "file_path": output_path
        }
        with open("processed_file.pkl", "wb") as f:
            pickle.dump(metadata, f)

        print("Metadata saved for evaluation script.")

    except Exception as e:
        print(f"Error processing PDF: {e}")


# Upload Widget
upload_widget = widgets.FileUpload(
    accept='.pdf',  # Accept only PDF files
    multiple=False  # Only allow single file upload
)


def on_upload_change(change):
    """Handle file upload and process PDF."""
    if upload_widget.value:
        for file_info in upload_widget.value:
            filename = file_info['name'].replace(".pdf", "")
            file_content = file_info['content'].tobytes()  # Convert memoryview to bytes
            
            # Process PDF
            preprocess_pdf(file_content, filename)


# Attach event listener
upload_widget.observe(on_upload_change, names='value')

display(upload_widget)

FileUpload(value=(), accept='.pdf', description='Upload')

In [3]:
import os
import pickle
import json
import random
import pandas as pd
import math
import re
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
import boto3
from botocore.config import Config
from dotenv import load_dotenv

#  Load Environment Variables
load_dotenv("codes.env")

aws_access_key = os.environ.get("AWS_ACCESS_KEY_ID")
aws_secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY")
aws_region = os.environ.get("AWS_REGION")

MODEL_ID_LLAMA = "us.meta.llama3-3-70b-instruct-v1:0"
config = Config(read_timeout=1000)

bedrock_client = boto3.client(
    "bedrock-runtime",
    region_name=aws_region,
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    config=config
)

#  Load Predefined Topics & Mappings
mapping_file_path = 'final_file_topic_mapping_v6.csv'
file_topic_mapping = pd.read_csv(mapping_file_path)
unique_topics = file_topic_mapping['folder_name'].unique().tolist()
unique_topics_str = ', '.join(unique_topics)

#  Load Trained Model & Vectorizer** 
print(" Loading Trained TF-IDF Vectorizer and Random Forest Model...")
tfidf_vectorizer = joblib.load("tfidf_vectorizer.pkl")
rf_model = joblib.load("rf_model.pkl")

#  Random Forest Classification** 
def rf_classify_document(text, confidence_threshold=0.99):
    """Classifies a document using the trained Random Forest model."""
    text_tfidf = tfidf_vectorizer.transform([text])  # Convert text to TF-IDF features
    y_pred_proba = rf_model.predict_proba(text_tfidf)[0]  # Get probability scores
    
    max_prob = max(y_pred_proba)
    predicted_topic = rf_model.classes_[y_pred_proba.argmax()]

    if max_prob < confidence_threshold:
        return None, max_prob  # Defer to LLM

    return predicted_topic, max_prob


#  Sampling Logic for Hybrid Text Extraction**
def extract_intro_middle_conclusion(text, max_tokens=20000):
    words = text.split()
    total_words = len(words)

    if total_words < 5000:
        ratios = (0.15, 0.15, 0.15)
    elif total_words < 20000:
        ratios = (0.08, 0.12, 0.08)
    elif total_words < 50000:
        ratios = (0.04, 0.08, 0.04)
    else:
        ratios = (0.02, 0.06, 0.02)

    intro_end = max(int(total_words * ratios[0]), 100)
    conclusion_start = max(int(total_words * (1 - ratios[2])), total_words - 100)

    middle = words[intro_end:conclusion_start]
    middle_sample_size = int(len(middle) * ratios[1])
    middle_sample = random.sample(middle, min(middle_sample_size, len(middle)))

    hybrid_text_words = words[:intro_end] + middle_sample + words[conclusion_start:]
    estimated_tokens = len(hybrid_text_words) * 1.3

    if estimated_tokens > max_tokens:
        allowed_words = int(max_tokens / 1.3)
        hybrid_text_words = hybrid_text_words[:allowed_words]

    return " ".join(hybrid_text_words)

#  LLM Fallback (Uses Sampled Text)**
def evaluate_topic_with_llama(file_content):
    try:
        prompt = f"""
        Analyze the following document sample and classify it into only one of these topics: {unique_topics_str}.
        After explaining your reasoning, clearly state the final topic at the end.

        Document:
        {file_content}

        Explanation: <Your explanation>
        Final Topic: <One of the topics from the list>
        """

        formatted_prompt = f"""
        <|begin_of_text|>
        <|start_header_id|>user<|end_header_id|>
        {prompt}
        <|eot_id|>
        <|start_header_id|>assistant<|end_header_id|>
        """

        response = bedrock_client.invoke_model(
            modelId=MODEL_ID_LLAMA,
            body=json.dumps({
                "prompt": formatted_prompt,
                "max_gen_len": 512,
                "temperature": 0,
            }),
            contentType="application/json"
        )

        response_body = json.loads(response['body'].read())
        response_text = response_body.get("generation", "").strip()

        match = re.search(r"Final Topic:\s*(.+)", response_text, re.IGNORECASE)
        predicted_topic = match.group(1).strip() if match else "Unknown"

        return predicted_topic, response_text

    except Exception as e:
        print(f"Error calling AWS Bedrock API: {e}")
        return "Unknown", "Error occurred during LLM call"

#  Main Pipeline (Single File Upload)**
def process_single_file():
    with open("processed_file.pkl", "rb") as f:
        metadata = pickle.load(f)

    file_path = metadata["file_path"]
    filename = metadata["filename"]

    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    sampled_text = extract_intro_middle_conclusion(content)
    predicted_topic, confidence = rf_classify_document(sampled_text)

    if predicted_topic:
        print(f" Random Forest Classification: {predicted_topic} (Confidence: {confidence:.2f})")
        return filename, predicted_topic, confidence, "-", "Random Forest Rule-Based Classification"

    # Fallback to LLM using Sampled Text (Hybrid)
    predicted_topic, explanation = evaluate_topic_with_llama(sampled_text)

    print(f" LLM Classification: {predicted_topic}")
    return filename, predicted_topic, "-", explanation, "LLM"

#  Run and Output Results**
if __name__ == "__main__":
    filename, predicted_topic, confidence, explanation, source = process_single_file()

    result = {
        "Filename": filename,
        "Predicted Topic": predicted_topic,
        "Confidence": confidence,
        "Explanation": explanation,
        "Source": source
    }
    print(json.dumps(result, indent=4))

 Loading Trained TF-IDF Vectorizer and Random Forest Model...
 LLM Classification: Unknown
{
    "Filename": "landing_the_economic_case_for_climate_action_with_decision_makers",
    "Predicted Topic": "Unknown",
    "Confidence": "-",
    "Explanation": "Explanation: The document provided is a comprehensive report on the economic case for climate action, highlighting the physical impacts of climate change, economic damages, and the business case for climate action. It emphasizes the need for sustained reductions in emissions to limit global warming and mitigate its effects. The report is the result of a collaboration between the Climatraces Lab at the University of Cambridge and the Boston Consulting Group. It discusses various aspects such as the costs and benefits of climate action, barriers to economically rational climate action, and the importance of creating a transparent and equitable transition to a low-carbon economy. The document also references the Paris Agreement and the ne

# Module 1: Addition of the ability to add new topics!
## look at outputs in console!!

order of buttons:
1. upload test document (which should be classified as sustainability),
2. Classify Test Document (but model is not trained on, plus there is no topic on sustainability therefore should give a topic in our current list)
3. enter new topic name ("Sustainability")
4. upload new topic's document under the Sustainability topic
5. Add New Topic (which retrains models and adds new topic in strings)
6. Classify Test Document again which will output the test document's topic as Sustainability!

Remove Topic:
1. type in topic to remove ("Sustainability")
2. click Remove Topic button
3. classify Test Document again which will output test document's topic as NOT Sustainability!

In [2]:
import os
import re
import io
import json
import random
import pickle
import shutil
import pandas as pd
import joblib
import boto3
import ipywidgets as widgets
from IPython.display import display
from pypdf import PdfReader
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from botocore.config import Config
from dotenv import load_dotenv

# =============================================================================
# 1. Document Processing Utilities
# =============================================================================

class DocumentProcessor:
    """Utility class for cleaning and preprocessing PDF documents."""
    
    @staticmethod
    def clean_text(text: str) -> str:
        """
        Clean text by removing non-ASCII characters, URLs, special characters,
        converting to lowercase, and normalizing whitespace.
        """
        text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # Remove non-ASCII characters
        text = re.sub(r'http\S+', '', text)           # Remove URLs
        text = re.sub(r'[^a-zA-Z\s]', '', text)        # Remove special characters and numbers
        text = text.lower()                           # Convert to lowercase
        text = re.sub(r'\s+', ' ', text).strip()       # Normalize whitespace
        text = re.sub(r'\n+', '\n', text)              # Normalize newlines
        return text

    @staticmethod
    def remove_stop_words(text: str) -> str:
        """
        Remove stop words from text using NLTK's English stop word list.
        """
        stop_words = set(stopwords.words('english'))
        words = text.split()
        filtered_words = [word for word in words if word not in stop_words]
        return ' '.join(filtered_words)

    @staticmethod
    def preprocess_pdf(file_content: bytes, filename: str) -> (str, str):
        """
        Extract text from the PDF, clean it, remove stop words,
        and save the cleaned text.
        Returns a tuple of the output file path and the cleaned text.
        """
        try:
            pdf_stream = io.BytesIO(file_content)
            reader = PdfReader(pdf_stream)
            extracted_text = ""
            for page in reader.pages:
                text = page.extract_text()
                if text:
                    extracted_text += text + "\n"

            cleaned_text = DocumentProcessor.clean_text(extracted_text)
            cleaned_text = DocumentProcessor.remove_stop_words(cleaned_text)

            # Save cleaned text to a .txt file in the current working directory
            output_path = os.path.join(os.getcwd(), f"{filename}.txt")
            with open(output_path, "w", encoding="utf-8") as text_file:
                text_file.write(cleaned_text)

            print(f"Processed text saved to: {output_path}")
            print(f"Preview:\n{cleaned_text[:500]}...")
            return output_path, cleaned_text

        except Exception as e:
            print(f"Error processing PDF: {e}")
            raise


class ModelManager:
    """
    Manages loading/saving of the mapping file and models,
    retraining, and classification.

    This version uses a mapping file named final_file_topic_mapping_v6.csv 
    with two columns:
      - folder_name
      - file_name
    Files are stored in a multi-level folder structure under Extracted_Sample_Data.
    For example, a file with folder_name "Annual Reports" may be found under
      Extracted_Sample_Data/Financial/Annual Reports/
    """
    def __init__(self, mapping_file_path: str = 'final_file_topic_mapping_v6.csv'):
        self.mapping_file_path = mapping_file_path
        self.vectorizer_path = "tfidf_vectorizer.pkl"
        self.model_path = "rf_model.pkl"
        self.unique_topics = []
        self.unique_topics_str = ""
        self.load_models()  # load initial models
        self.update_unique_topics()  # load topics from mapping file

    @staticmethod
    def find_folder_recursively(root: str, target: str) -> str:
        """
        Recursively search for a directory named target under the root folder.
        Returns the full path if found; otherwise, returns None.
        """
        for dirpath, dirnames, _ in os.walk(root):
            # Compare folder names case-insensitively
            if os.path.basename(dirpath).lower() == target.lower():
                return dirpath
        return None

    def load_mapping_file(self) -> pd.DataFrame:
        """
        Load the topic mapping CSV file. If not found, return an empty DataFrame.
        Expected columns: file_name, folder_name.
        """
        if os.path.exists(self.mapping_file_path):
            return pd.read_csv(self.mapping_file_path)
        else:
            return pd.DataFrame(columns=["folder_name", "file_name"])

    def save_mapping_file(self, df: pd.DataFrame) -> None:
        """Save the mapping DataFrame to a CSV file."""
        df.to_csv(self.mapping_file_path, index=False)

    def update_unique_topics(self) -> None:
        """
        Load the mapping file and update the unique topics list and corresponding string.
        """
        df = self.load_mapping_file()
        df = df.dropna(subset=["folder_name"])
        self.unique_topics = df['folder_name'].unique().tolist()
        self.unique_topics_str = ', '.join(self.unique_topics)
        print("Loaded topics:", self.unique_topics_str)

    def update_mapping_paths(self) -> pd.DataFrame:
        """
        Loop through mapping file rows and verify file existence.
        Files are expected to be located in a folder found recursively under Extracted_Sample_Data.
        This function reports any missing files.
        """
        df = self.load_mapping_file()
        for idx, row in df.iterrows():
            folder = row['folder_name']
            file_name = row['file_name']
            folder_path = self.find_folder_recursively("Extracted_Sample_Data", folder)
            if folder_path:
                full_path = os.path.join(folder_path, file_name)
            else:
                full_path = os.path.join("Extracted_Sample_Data", folder, file_name)
            if not os.path.exists(full_path):
                print(f"Warning: File not found for row {idx}: {full_path}")
        return df

    def retrain_model(self) -> None:
        """
        Retrain the TF-IDF vectorizer and Random Forest classifier using all
        documents listed in the mapping CSV.
        """
        # Check mapping paths (this version reports missing files)
        df = self.update_mapping_paths()
        # Drop rows with missing folder or file names
        df = df.dropna(subset=["folder_name", "file_name"])
        texts = []
        labels = []
        for _, row in df.iterrows():
            folder = row['folder_name']
            file_name = row['file_name']
            folder_path = self.find_folder_recursively("Extracted_Sample_Data", folder)
            if folder_path:
                full_path = os.path.join(folder_path, file_name)
            else:
                full_path = os.path.join("Extracted_Sample_Data", folder, file_name)
            try:
                with open(full_path, 'r', encoding="utf-8") as f:
                    text = f.read()
                texts.append(text)
                labels.append(folder)
            except Exception as e:
                print(f"Error reading file {full_path}: {e}")

        if not texts:
            print("No training data available. Skipping retraining.")
            return

        # Train new TF-IDF vectorizer and Random Forest classifier
        vectorizer = TfidfVectorizer()
        X = vectorizer.fit_transform(texts)
        classifier = RandomForestClassifier(random_state=42)
        classifier.fit(X, labels)

        # Save updated models
        joblib.dump(vectorizer, self.vectorizer_path)
        joblib.dump(classifier, self.model_path)
        print("Retrained and updated model and vectorizer.")

        # Update topics list from the mapping file
        df = self.load_mapping_file()
        self.unique_topics = df['folder_name'].unique().tolist()
        self.unique_topics_str = ', '.join(self.unique_topics)
        print("Updated topics list:", self.unique_topics_str)
        # Refresh in-memory models
        self.load_models()

    def add_new_topic(self, topic_name: str, file_content: bytes, original_filename: str, processor) -> None:
        """
        Add a new topic by processing the provided PDF document, moving the file to the correct
        subfolder under Extracted_Sample_Data, updating the mapping CSV, and retraining the model.
        """
        # Process the PDF document
        output_path, _ = processor.preprocess_pdf(file_content, original_filename)
        # Determine destination folder: create it under Extracted_Sample_Data if needed
        # Here we assume that new topics are created directly under Extracted_Sample_Data.
        dest_folder = os.path.join("Extracted_Sample_Data", topic_name)
        os.makedirs(dest_folder, exist_ok=True)
        new_destination = os.path.join(dest_folder, os.path.basename(output_path))
        shutil.move(output_path, new_destination)
        # Update mapping file with the new entry using file_name (not full path)
        df = self.load_mapping_file()
        new_entry = {"folder_name": topic_name, "file_name": os.path.basename(new_destination)}
        df = pd.concat([df, pd.DataFrame([new_entry])], ignore_index=True)
        self.save_mapping_file(df)
        print(f"New topic '{topic_name}' added with document: {new_destination}")
        self.retrain_model()
        self.update_unique_topics()

    def remove_topic(self, topic_name: str) -> None:
        """
        Remove an existing topic by deleting all related files in the corresponding folder under
        Extracted_Sample_Data, updating the mapping CSV, and retraining the model.
        Validates whether the topic exists before removal.
        """
        existing_topic = None
        for topic in self.unique_topics:
            if topic.lower() == topic_name.lower():
                existing_topic = topic
                break

        if not existing_topic:
            print(f"Topic '{topic_name}' does not exist.")
            return

        # Remove the topic folder and its contents
        topic_folder_path = os.path.join("Extracted_Sample_Data", existing_topic)
        if os.path.exists(topic_folder_path):
            try:
                shutil.rmtree(topic_folder_path)
                print(f"Deleted folder: {topic_folder_path}")
            except Exception as e:
                print(f"Error deleting folder {topic_folder_path}: {e}")
                return
        else:
            print(f"Folder for topic '{existing_topic}' not found.")

        # Update mapping CSV by removing entries for this topic
        df = self.load_mapping_file()
        original_count = len(df)
        df = df[df['folder_name'].str.lower() != existing_topic.lower()]
        updated_count = len(df)
        if original_count != updated_count:
            self.save_mapping_file(df)
            print(f"Removed {original_count - updated_count} mapping entries for topic '{existing_topic}'.")
        else:
            print("No mapping entries found for the topic.")

        self.retrain_model()
        self.update_unique_topics()
        print(f"Topic '{existing_topic}' has been successfully removed.")

    def load_models(self) -> None:
        """
        Load the trained TF-IDF vectorizer and Random Forest classifier.
        """
        try:
            self.tfidf_vectorizer = joblib.load(self.vectorizer_path)
            self.rf_model = joblib.load(self.model_path)
            print("Models loaded successfully.")
        except Exception as e:
            print("Error loading models. Make sure the model files exist.", e)
            self.tfidf_vectorizer, self.rf_model = None, None

    def classify_document(self) -> tuple[str, str, float, str, str]:
        """
        Classify a document using the uploaded test document.
        Returns a tuple: (filename, predicted_topic, confidence, explanation, source).
        Note: The test document metadata is stored separately (with full file path).
        """
        with open("processed_file.pkl", "rb") as f:
            metadata = pickle.load(f)
        file_path = metadata["file_path"]
        filename = metadata["filename"]

        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()

        sampled_text = self.extract_intro_middle_conclusion(content)
        predicted_topic, confidence = self.rf_classify_document(sampled_text)

        if predicted_topic:
            print(f"Random Forest Classification: {predicted_topic} (Confidence: {confidence:.2f})")
            return filename, predicted_topic, confidence, "-", "Random Forest Classification"

        predicted_topic, explanation = self.evaluate_topic_with_llama(sampled_text)
        print(f"LLM Classification: {predicted_topic}")
        return filename, predicted_topic, -1, explanation, "LLM Classification"

    def rf_classify_document(self, text: str, confidence_threshold: float = 0.99) -> tuple[str, float]:
        text_tfidf = self.tfidf_vectorizer.transform([text])
        y_pred_proba = self.rf_model.predict_proba(text_tfidf)[0]
        max_prob = max(y_pred_proba)
        predicted_topic = self.rf_model.classes_[y_pred_proba.argmax()]
        if max_prob < confidence_threshold:
            return None, max_prob
        return predicted_topic, max_prob

    def extract_intro_middle_conclusion(self, text: str, max_tokens: int = 20000) -> str:
        words = text.split()
        total_words = len(words)
        if total_words < 5000:
            ratios = (0.15, 0.15, 0.15)
        elif total_words < 20000:
            ratios = (0.08, 0.12, 0.08)
        elif total_words < 50000:
            ratios = (0.04, 0.08, 0.04)
        else:
            ratios = (0.02, 0.06, 0.02)
        intro_end = max(int(total_words * ratios[0]), 100)
        conclusion_start = max(int(total_words * (1 - ratios[2])), total_words - 100)
        middle = words[intro_end:conclusion_start]
        middle_sample_size = int(len(middle) * ratios[1])
        middle_sample = random.sample(middle, min(middle_sample_size, len(middle)))
        hybrid_text_words = words[:intro_end] + middle_sample + words[conclusion_start:]
        estimated_tokens = len(hybrid_text_words) * 1.3
        if estimated_tokens > max_tokens:
            allowed_words = int(max_tokens / 1.3)
            hybrid_text_words = hybrid_text_words[:allowed_words]
        return " ".join(hybrid_text_words)

    def evaluate_topic_with_llama(self, text: str) -> tuple[str, str]:
        """
        Use AWS Bedrock (LLM) as a fallback to classify the document if the classifier's confidence is low.
        This method uses the preloaded unique_topics_str.
        """
        try:
            prompt = f"""
            Analyze the following document sample and classify it into only one of these topics: {self.unique_topics_str}.
            After explaining your reasoning, clearly state the final topic at the end.
            
            Document:
            {text}
            
            Explanation: <Your explanation>
            Final Topic: <One of the topics from the list>
            """
            formatted_prompt = f"""
            <|begin_of_text|>
            <|start_header_id|>user<|end_header_id|>
            {prompt}
            <|eot_id|>
            <|start_header_id|>assistant<|end_header_id|>
            """
            load_dotenv("codes.env")
            aws_access_key = os.environ.get("AWS_ACCESS_KEY_ID")
            aws_secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY")
            aws_region = os.environ.get("AWS_REGION")
            MODEL_ID_LLAMA = "us.meta.llama3-3-70b-instruct-v1:0"
            config = Config(read_timeout=1000)
            bedrock_client = boto3.client(
                "bedrock-runtime",
                region_name=aws_region,
                aws_access_key_id=aws_access_key,
                aws_secret_access_key=aws_secret_key,
                config=config
            )
            response = bedrock_client.invoke_model(
                modelId=MODEL_ID_LLAMA,
                body=json.dumps({
                    "prompt": formatted_prompt,
                    "max_gen_len": 512,
                    "temperature": 0,
                }),
                contentType="application/json"
            )
            response_body = json.loads(response['body'].read())
            response_text = response_body.get("generation", "").strip()
            match = re.search(r"Final Topic:\s*(.+)", response_text, re.IGNORECASE)
            predicted_topic = match.group(1).strip() if match else "Unknown"
            return predicted_topic, response_text
        except Exception as e:
            print(f"Error calling AWS Bedrock API: {e}")
            return "Unknown", "Error occurred during LLM call"

# =============================================================================
# 3. Interactive Application Using ipywidgets
# =============================================================================

class DocumentClassifierApp:
    """
    Interactive application that manages the sequential workflow:
      1. Upload a test document for classification.
      2. Classify the test document.
      3. Add a new topic (with associated training document).
      4. Re-classify the test document to see the update.
    """
    def __init__(self):
        self.processor = DocumentProcessor()
        self.model_manager = ModelManager()
        self.setup_widgets()

    def setup_widgets(self):
        # Widget for test document upload (for classification)
        self.test_upload = widgets.FileUpload(
            accept='.pdf',  # only PDF files
            multiple=False
        )
        self.classify_button = widgets.Button(
            description='Classify Test Document'
        )
        self.classify_button.on_click(self.on_classify_click)
        
        # Widget group for adding a new topic
        self.new_topic_text = widgets.Text(
            description='Topic Name:',
            placeholder='Enter new topic name'
        )
        self.new_topic_upload = widgets.FileUpload(
            accept='.pdf',
            multiple=True # allow multiple file uploads
        )
        self.add_topic_button = widgets.Button(
            description='Add New Topic'
        )
        self.add_topic_button.on_click(self.on_add_topic_click)

        # Widgets for removing an existing topic
        self.remove_topic_text = widgets.Text(
            description='Remove Topic:',
            placeholder='Enter topic name to remove'
        )
        self.remove_topic_button = widgets.Button(
            description='Remove Topic'
        )
        self.remove_topic_button.on_click(self.on_remove_topic_click)

        # Display the widgets in a vertical layout
        display(
            widgets.VBox([
                widgets.Label("Upload Test Document for Classification"),
                self.test_upload,
                self.classify_button,
                widgets.Label("Add New Topic (with Training Document)"),
                self.new_topic_text,
                self.new_topic_upload,
                self.add_topic_button,
                widgets.Label("Remove an Existing Topic"),
                self.remove_topic_text,
                self.remove_topic_button
            ])
        )

        # Attach a callback for when a test document is uploaded:
        self.test_upload.observe(self.on_test_upload_change, names='value')

    def on_test_upload_change(self, change):
        """Process test document upload and save metadata for later classification."""
        if self.test_upload.value:
            for file_info in self.test_upload.value:
                filename = file_info['name'].replace(".pdf", "")
                file_content = file_info['content'].tobytes()
                # Process the PDF and save the cleaned text
                output_path, _ = self.processor.preprocess_pdf(file_content, filename)
                # Save metadata to be used by the classification function
                metadata = {"filename": filename, "file_path": output_path}
                with open("processed_file.pkl", "wb") as f:
                    pickle.dump(metadata, f)
                print("Test document processed and metadata saved.")

    def on_add_topic_click(self, b):
        """Handle the 'Add New Topic' button click event."""
        if self.new_topic_upload.value and self.new_topic_text.value:
            topic_name = self.new_topic_text.value.strip()
            for file_info in self.new_topic_upload.value:
                filename = file_info['name'].replace(".pdf", "")
                file_content = file_info['content'].tobytes()
                self.model_manager.add_new_topic(topic_name, file_content, filename, self.processor)
            # Reset input widgets
            self.new_topic_text.value = ""
            # self.new_topic_upload.value.clear()
            self.new_topic_upload._counter = 0  # Reset upload widget state
        else:
            print("Please provide both a topic name and a PDF document.")

    def on_remove_topic_click(self, b):
        """Handle the 'Remove Topic' button click event."""
        topic_name = self.remove_topic_text.value.strip()
        if topic_name:
            self.model_manager.remove_topic(topic_name)
            self.remove_topic_text.value = ""
        else:
            print("Please provide a topic name to remove.")

    def on_classify_click(self, b):
        """Handle the 'Classify Test Document' button click event."""
        print("Classify button clicked!")  # Debugging print

        try:
            filename, predicted_topic, confidence, explanation, source = self.model_manager.classify_document()

            result = {
                "Filename": filename,
                "Predicted Topic": predicted_topic,
                "Confidence": confidence,
                "Explanation": explanation,
                "Source": source
            }
            print("Classification Result:")
            print(json.dumps(result, indent=4))
        except Exception as e:
            print("Error during classification:", e)

# Initialize the interactive app
app = DocumentClassifierApp()

Models loaded successfully.
Loaded topics: Marketing and Public Communication, Annual Reports, Investment and Market Research, Taxation, Risk Management, Audit Reports, Consumer Finance, Financial Regulations, Anti Money Laundering, Technology, Loans, Mergers and Acquisitions, Derivatives, Employment, Non-Disclosure Agreements (NDA), Partnerships, Client Agreements, Strategic, Administrative, Operational, Sustainability


# Module 2: Feedback Retraining Module
The code in this section handles the output from the UI when the user has selected the documents to send for retraining.

In [ ]:
import os
import pandas as pd
import shutil
import random
from pypdf import PdfReader
import re
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
import joblib
import time

class DocumentProcessor:
    """Utility class for cleaning and preprocessing PDF documents."""
    @staticmethod
    def clean_text(text: str) -> str:
        text = re.sub(r'[^\x00-\x7F]+', ' ', text)
        text = re.sub(r'http\S+', '', text)
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        text = text.lower()
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    @staticmethod
    def remove_stop_words(text: str) -> str:
        stop_words = set(stopwords.words('english'))
        words = text.split()
        return ' '.join(word for word in words if word not in stop_words)

    def preprocess_pdf(self, file_path: str) -> str:
        try:
            reader = PdfReader(file_path)
            extracted_text = ""
            for page in reader.pages:
                text = page.extract_text()
                if text:
                    extracted_text += text + "\n"
            cleaned_text = self.clean_text(extracted_text)
            return self.remove_stop_words(cleaned_text)
        except Exception as e:
            print(f"Error processing {file_path}: {e}")
            return None

class ModelManager:
    """Manages model training, updating, and document classification."""
    def __init__(self):
        self.document_dir = 'Extracted_Sample_Data'
        self.vectorizer = joblib.load('tfidf_vectorizer.pkl')
        self.model = joblib.load('rf_model.pkl')
        self.feedback_df = pd.read_csv('user_feedback.csv')
        self.mapping_df = pd.read_csv('final_file_topic_mapping_v6.csv')
        self.processor = DocumentProcessor()

    def update_document_locations(self):
        """Update document locations based on feedback and preprocess new documents."""
        for index, row in self.feedback_df.iterrows():
            file_name = row['name']
            corrected_topic = row['user_corrected_category']
            search_name = file_name.split('.')[0]
            file_path = next((os.path.join(root, f) for root, dirs, files in os.walk(self.document_dir)
                              for f in files if f.startswith(search_name) and f.endswith('.txt')), None)
            if file_path:
                self.move_file(file_path, corrected_topic)
            else:
                self.process_and_move_new_file(file_name, corrected_topic)

    def move_file(self, file_path, corrected_topic):
        current_directory = os.path.dirname(file_path)
        correct_directory = os.path.join(self.document_dir, corrected_topic)
        if not os.path.exists(correct_directory):
            os.makedirs(correct_directory, exist_ok=True)
        new_file_path = os.path.join(correct_directory, os.path.basename(file_path))
        shutil.move(file_path, new_file_path)
        print(f"Moved '{os.path.basename(file_path)}' to '{correct_directory}'")

    def process_and_move_new_file(self, file_name, corrected_topic):
        file_path_pdf = os.path.join(self.document_dir, file_name)
        processed_text = self.processor.preprocess_pdf(file_path_pdf)
        if processed_text:
            processed_text = self.extract_intro_middle_conclusion(processed_text)
            file_path_txt = file_path_pdf.replace('.pdf', '_extracted.txt')
            with open(file_path_txt, 'w', encoding='utf-8') as file:
                file.write(processed_text)
            self.move_file(file_path_txt, corrected_topic)

    def extract_intro_middle_conclusion(self, text: str, max_tokens: int = 20000) -> str:
        words = text.split()
        total_words = len(words)
        if total_words < 5000:
            ratios = (0.15, 0.15, 0.15)
        elif total_words < 20000:
            ratios = (0.08, 0.12, 0.08)
        elif total_words < 50000:
            ratios = (0.04, 0.08, 0.04)
        else:
            ratios = (0.02, 0.06, 0.02)
        intro_end = max(int(total_words * ratios[0]), 100)
        conclusion_start = max(int(total_words * (1 - ratios[2])), total_words - 100)
        middle = words[intro_end:conclusion_start]
        middle_sample_size = int(len(middle) * ratios[1])
        middle_sample = random.sample(middle, min(middle_sample_size, len(middle)))
        hybrid_text_words = words[:intro_end] + middle_sample + words[conclusion_start:]
        estimated_tokens = len(hybrid_text_words) * 1.3
        if estimated_tokens > max_tokens:
            allowed_words = int(max_tokens / 1.3)
            hybrid_text_words = hybrid_text_words[:allowed_words]
        return " ".join(hybrid_text_words)

    def train_model(self):
        """Train the model using all documents."""
        documents, labels = [], []
        for root, _, files in os.walk(self.document_dir):
            for file in files:
                if file.endswith(".txt"):
                    file_path = os.path.join(root, file)
                    with open(file_path, 'r', encoding='utf-8') as f:
                        documents.append(f.read())
                    labels.append(root.split('/')[-1])

        X = self.vectorizer.fit_transform(documents)
        y = pd.Series(labels)

        smote_enn = SMOTEENN(smote=SMOTE(random_state=42), random_state=42)
        X_resampled, y_resampled = smote_enn.fit_resample(X, y)

        rf_model = RandomForestClassifier(
            n_estimators=5000, max_depth=30, min_samples_split=2,
            min_samples_leaf=2, class_weight="balanced", random_state=42, n_jobs=-1
        )
        rf_model.fit(X_resampled, y_resampled)

        calibrated_rf = CalibratedClassifierCV(rf_model, method='isotonic', cv='prefit')
        calibrated_rf.fit(X_resampled, y_resampled)

        timestamp = time.strftime("%Y%m%d_%H%M%S")
        model_filename = f"rf_model_{timestamp}.pkl"
        joblib.dump(calibrated_rf, model_filename)
        print(f"New model saved as {model_filename}")

# Example of using the classes
manager = ModelManager()
manager.update_document_locations()
manager.train_model()


# Final Integrated Code with Both Modules

In [ ]:
import os
import pandas as pd
import shutil
import random
import io
import json
import pickle
import boto3
import ipywidgets as widgets
from IPython.display import display
from pypdf import PdfReader
import re
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from scipy.sparse import vstack
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
import joblib
import time
from dotenv import load_dotenv
from botocore.config import Config

class DocumentProcessor:
    """Utility class for cleaning and preprocessing PDF documents."""
    @staticmethod
    def clean_text(text: str) -> str:
        text = re.sub(r'[^\x00-\x7F]+', ' ', text)
        text = re.sub(r'http\S+', '', text)
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        text = text.lower()
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    @staticmethod
    def remove_stop_words(text: str) -> str:
        stop_words = set(stopwords.words('english'))
        words = text.split()
        return ' '.join(word for word in words if word not in stop_words)

    def preprocess_pdf(self, file_content: bytes, filename: str) -> tuple[str, str]:
        try:
            pdf_stream = io.BytesIO(file_content)
            reader = PdfReader(pdf_stream)
            extracted_text = ""
            for page in reader.pages:
                text = page.extract_text()
                if text:
                    extracted_text += text + "\n"
            cleaned_text = self.clean_text(extracted_text)
            final_text = self.remove_stop_words(cleaned_text)
            output_path = os.path.join(os.getcwd(), f"{filename}_extracted.txt")
            with open(output_path, "w", encoding="utf-8") as text_file:
                text_file.write(final_text)
            return output_path, final_text
        except Exception as e:
            print(f"Error processing PDF: {e}")
            return None, None

class ModelManager:
    """Manages model training, updating, and document classification."""
    def __init__(self,mapping_file_path: str = 'final_file_topic_mapping_v6.csv'):
        self.document_dir = 'Extracted_Sample_Data'
        self.vectorizer_path = "tfidf_vectorizer.pkl"
        self.model_path = 'rf_model.pkl'
        self.feedback_df = pd.read_csv('user_feedback.csv')
        self.mapping_file_path = mapping_file_path

        self.unique_topics = []
        self.unique_topics_str = ""
        self.load_models()  # load initial models
        self.update_unique_topics()  # load topics from mapping file
        self.processor = DocumentProcessor()

    def load_models(self) -> None:
        """
        Load the trained TF-IDF vectorizer and Random Forest classifier.
        """
        try:
            self.tfidf_vectorizer = joblib.load(self.vectorizer_path)
            self.rf_model = joblib.load(self.model_path)
            print("Models loaded successfully.")
        except Exception as e:
            print("Error loading models. Make sure the model files exist.", e)
            self.tfidf_vectorizer, self.rf_model = None, None
            
    def extract_intro_middle_conclusion(self, text: str, max_tokens: int = 20000) -> str:
        words = text.split()
        total_words = len(words)
        if total_words < 5000:
            ratios = (0.15, 0.15, 0.15)
        elif total_words < 20000:
            ratios = (0.08, 0.12, 0.08)
        elif total_words < 50000:
            ratios = (0.04, 0.08, 0.04)
        else:
            ratios = (0.02, 0.06, 0.02)
        intro_end = max(int(total_words * ratios[0]), 100)
        conclusion_start = max(int(total_words * (1 - ratios[2])), total_words - 100)
        middle = words[intro_end:conclusion_start]
        middle_sample_size = int(len(middle) * ratios[1])
        middle_sample = random.sample(middle, min(middle_sample_size, len(middle)))
        hybrid_text_words = words[:intro_end] + middle_sample + words[conclusion_start:]
        estimated_tokens = len(hybrid_text_words) * 1.3
        if estimated_tokens > max_tokens:
            allowed_words = int(max_tokens / 1.3)
            hybrid_text_words = hybrid_text_words[:allowed_words]
        return " ".join(hybrid_text_words)
    
    @staticmethod
    def find_folder_recursively(root: str, target: str) -> str:
        """
        Recursively search for a directory named target under the root folder.
        Returns the full path if found; otherwise, returns None.
        """
        for dirpath, dirnames, _ in os.walk(root):
            # Compare folder names case-insensitively
            if os.path.basename(dirpath).lower() == target.lower():
                return dirpath
        return None

    def load_mapping_file(self) -> pd.DataFrame:
        """
        Load the topic mapping CSV file. If not found, return an empty DataFrame.
        Expected columns: file_name, folder_name.
        """
        if os.path.exists(self.mapping_file_path):
            return pd.read_csv(self.mapping_file_path)
        else:
            return pd.DataFrame(columns=["folder_name", "file_name"])

    def update_unique_topics(self) -> None:
        """
        Load the mapping file and update the unique topics list and corresponding string.
        """
        df = self.load_mapping_file()
        df = df.dropna(subset=["folder_name"])
        self.unique_topics = df['folder_name'].unique().tolist()
        self.unique_topics_str = ', '.join(self.unique_topics)
        print("Loaded topics:", self.unique_topics_str)

    def save_mapping_file(self, df: pd.DataFrame) -> None:
        """Save the mapping DataFrame to a CSV file."""
        df.to_csv(self.mapping_file_path, index=False)
    def classify_document(self) -> tuple[str, str, float, str, str]:
        """
        Classify a document using the uploaded test document.
        Returns a tuple: (filename, predicted_topic, confidence, explanation, source).
        Note: The test document metadata is stored separately (with full file path).
        """
        with open("processed_file.pkl", "rb") as f:
            metadata = pickle.load(f)
        file_path = metadata["file_path"]
        filename = metadata["filename"]

        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()

        sampled_text = self.extract_intro_middle_conclusion(content)
        predicted_topic, confidence = self.rf_classify_document(sampled_text)

        if predicted_topic:
            print(f"Random Forest Classification: {predicted_topic} (Confidence: {confidence:.2f})")
            return filename, predicted_topic, confidence, "-", "Random Forest Classification"

        predicted_topic, explanation = self.evaluate_topic_with_llama(sampled_text)
        print(f"LLM Classification: {predicted_topic}")
        return filename, predicted_topic, "-", explanation, "LLM Classification"

    def rf_classify_document(self, text: str, confidence_threshold: float = 0.99) -> tuple[str, float]:
        text_tfidf = self.tfidf_vectorizer.transform([text])
        y_pred_proba = self.rf_model.predict_proba(text_tfidf)[0]
        max_prob = max(y_pred_proba)
        predicted_topic = self.rf_model.classes_[y_pred_proba.argmax()]
        if max_prob < confidence_threshold:
            return None, max_prob
        return predicted_topic, max_prob
    
    def evaluate_topic_with_llama(self, text: str) -> tuple[str, str]:
        """
        Use AWS Bedrock (LLM) as a fallback to classify the document if the classifier's confidence is low.
        This method uses the preloaded unique_topics_str.
        """
        try:
            prompt = f"""
            Analyze the following document sample and classify it into only one of these topics: {self.unique_topics_str}.
            After explaining your reasoning, clearly state the final topic at the end.
            
            Document:
            {text}
            
            Explanation: <Your explanation>
            Final Topic: <One of the topics from the list>
            """
            formatted_prompt = f"""
            <|begin_of_text|>
            <|start_header_id|>user<|end_header_id|>
            {prompt}
            <|eot_id|>
            <|start_header_id|>assistant<|end_header_id|>
            """
            load_dotenv("codes.env")
            aws_access_key = os.environ.get("AWS_ACCESS_KEY_ID")
            aws_secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY")
            aws_region = os.environ.get("AWS_REGION")
            MODEL_ID_LLAMA = "us.meta.llama3-3-70b-instruct-v1:0"
            config = Config(read_timeout=1000)
            bedrock_client = boto3.client(
                "bedrock-runtime",
                region_name=aws_region,
                aws_access_key_id=aws_access_key,
                aws_secret_access_key=aws_secret_key,
                config=config
            )
            response = bedrock_client.invoke_model(
                modelId=MODEL_ID_LLAMA,
                body=json.dumps({
                    "prompt": formatted_prompt,
                    "max_gen_len": 512,
                    "temperature": 0,
                }),
                contentType="application/json"
            )
            response_body = json.loads(response['body'].read())
            response_text = response_body.get("generation", "").strip()
            match = re.search(r"Final Topic:\s*(.+)", response_text, re.IGNORECASE)
            predicted_topic = match.group(1).strip() if match else "Unknown"
            return predicted_topic, response_text
        except Exception as e:
            print(f"Error calling AWS Bedrock API: {e}")
            return "Unknown", "Error occurred during LLM call"
    
    def handle_user_feedback(self):
        """Handle user feedback to update document locations and retrain the model as necessary."""
        for index, row in self.feedback_df.iterrows():
            file_name = row['name']
            corrected_topic = row['user_corrected_category']
            search_name = file_name.split('.')[0]

            file_path = next((os.path.join(root, f) for root, dirs, files in os.walk(self.document_dir)
                            for f in files if f.startswith(search_name) and f.endswith('.txt')), None)
            if file_path:
                self.move_file(file_path, corrected_topic)
            else:
                self.process_and_move_new_file(file_name, corrected_topic)

        self.retrain_model()

        self.update_mapping_csv()
        print("User feedback processed and model updated.")

    def move_file(self, file_path, corrected_topic):
        current_directory = os.path.dirname(file_path)
        correct_directory = os.path.join(self.document_dir, corrected_topic)
        if not os.path.exists(correct_directory):
            os.makedirs(correct_directory, exist_ok=True)
        new_file_path = os.path.join(correct_directory, os.path.basename(file_path))
        shutil.move(file_path, new_file_path)
        print(f"Moved '{os.path.basename(file_path)}' to '{correct_directory}'")

    def process_and_move_new_file(self, file_name, corrected_topic):
        file_path_pdf = os.path.join(self.document_dir, file_name)
        _, processed_text = self.processor.preprocess_pdf(file_path_pdf)
        if processed_text:
            processed_text = self.extract_intro_middle_conclusion(processed_text)
            file_path_txt = file_path_pdf.replace('.pdf', '_extracted.txt')
            with open(file_path_txt, 'w', encoding='utf-8') as file:
                file.write(processed_text)
            self.move_file(file_path_txt, corrected_topic)

    def retrain_model(self):
        """Train the model using all documents."""
        documents, labels = [], []
        for root, _, files in os.walk(self.document_dir):
            for file in files:
                if file.endswith(".txt"):
                    file_path = os.path.join(root, file)
                    with open(file_path, 'r', encoding="utf-8") as f:
                        documents.append(f.read())
                    labels.append(root.split('/')[-1])

        X = self.tfidf_vectorizer.fit_transform(documents)
        y = pd.Series(labels)

         # Identify classes with at least two samples
        class_counts = y.value_counts()
        sufficient_samples = class_counts[class_counts >= 2].index.tolist()
        insufficient_samples = class_counts[class_counts < 2].index.tolist()

        # Filter data for SMOTE
        sufficient_mask = y.isin(sufficient_samples)
        sufficient_indices = sufficient_mask.values.nonzero()[0]  # Convert boolean mask to indices
        X_sufficient = X[sufficient_indices]
        y_sufficient = y.iloc[sufficient_indices]

        # Apply SMOTE only to classes with sufficient samples
        smote = SMOTE(random_state=42, k_neighbors=1)
        smote_enn = SMOTEENN(smote=smote, random_state=42)
        X_resampled, y_resampled = smote_enn.fit_resample(X_sufficient, y_sufficient)

        # Combine resampled data with the insufficient samples data
        if insufficient_samples:
            insufficient_indices = (~sufficient_mask.values).nonzero()[0]
            X_insufficient = X[insufficient_indices]
            y_insufficient = y.iloc[insufficient_indices]
            X_resampled = vstack([X_resampled, X_insufficient])
            y_resampled = pd.concat([y_resampled, y_insufficient])


        # Train the random forest classifier
        rf_model = RandomForestClassifier(
            n_estimators=5000, max_depth=30, min_samples_split=2,
            min_samples_leaf=2, class_weight="balanced", random_state=42, n_jobs=-1
        )
        rf_model.fit(X_resampled, y_resampled)

        # Calibrate model
        calibrated_rf = CalibratedClassifierCV(rf_model, method='isotonic', cv='prefit')
        calibrated_rf.fit(X_resampled, y_resampled)

        # Save the retrained model
        model_filename = "rf_model.pkl"
        joblib.dump(self.tfidf_vectorizer, 'tfidf_vectorizer.pkl')
        joblib.dump(calibrated_rf, model_filename)
        print(f"New model saved as {model_filename}")

    def update_mapping_csv(self): 
        """Update the mapping file based on current folder structures."""
        new_rows = []
        for root, dirs, files in os.walk(self.document_dir):
            for file in files:
                if file.endswith('.txt'):
                    folder_name = os.path.basename(root)
                    new_rows.append({'file_name': file, 'folder_name': folder_name})
        
        new_df = pd.DataFrame(new_rows)
        new_df = new_df[['file_name', 'folder_name']]  
        new_df.to_csv(self.mapping_file_path, index=False)
        print("Updated mapping.csv file.")

    def add_new_topic(self, topic_name: str, file_content: bytes, original_filename: str, processor) -> None:
        """
        Add a new topic by processing the provided PDF document, moving the file to the correct
        subfolder under Extracted_Sample_Data, updating the mapping CSV, and retraining the model.
        """
        # Process the PDF document
        output_path, _ = processor.preprocess_pdf(file_content, original_filename)
        # Determine destination folder: create it under Extracted_Sample_Data if needed
        # Here we assume that new topics are created directly under Extracted_Sample_Data.
        dest_folder = os.path.join("Extracted_Sample_Data", topic_name)
        os.makedirs(dest_folder, exist_ok=True)
        new_destination = os.path.join(dest_folder, os.path.basename(output_path))
        shutil.move(output_path, new_destination)
        # Update mapping file with the new entry using file_name (not full path)
        df = self.load_mapping_file()
        new_entry = {"folder_name": topic_name, "file_name": os.path.basename(new_destination)}
        df = pd.concat([df, pd.DataFrame([new_entry])], ignore_index=True)
        self.save_mapping_file(df)
        print(f"New topic '{topic_name}' added with document: {new_destination}")
        self.retrain_model()
        self.update_unique_topics()

    def remove_topic(self, topic_name: str) -> None:
        """
        Remove an existing topic by deleting all related files in the corresponding folder under
        Extracted_Sample_Data, updating the mapping CSV, and retraining the model.
        Validates whether the topic exists before removal.
        """
        existing_topic = None
        for topic in self.unique_topics:
            if topic.lower() == topic_name.lower():
                existing_topic = topic
                break

        if not existing_topic:
            print(f"Topic '{topic_name}' does not exist.")
            return

        # Remove the topic folder and its contents
        topic_folder_path = os.path.join("Extracted_Sample_Data", existing_topic)
        if os.path.exists(topic_folder_path):
            try:
                shutil.rmtree(topic_folder_path)
                print(f"Deleted folder: {topic_folder_path}")
            except Exception as e:
                print(f"Error deleting folder {topic_folder_path}: {e}")
                return
        else:
            print(f"Folder for topic '{existing_topic}' not found.")

        # Update mapping CSV by removing entries for this topic
        df = self.load_mapping_file()
        original_count = len(df)
        df = df[df['folder_name'].str.lower() != existing_topic.lower()]
        updated_count = len(df)
        if original_count != updated_count:
            self.save_mapping_file(df)
            print(f"Removed {original_count - updated_count} mapping entries for topic '{existing_topic}'.")
        else:
            print("No mapping entries found for the topic.")

        self.retrain_model()
        self.update_unique_topics()
        print(f"Topic '{existing_topic}' has been successfully removed.")

    
class DocumentClassifierApp:
    """Interactive application for managing document classification and topic management."""
    def __init__(self):
        self.processor = DocumentProcessor()
        self.model_manager = ModelManager()
        self.setup_widgets()

    def setup_widgets(self):
        """Setup interactive widgets for document and topic management."""
        self.test_upload = widgets.FileUpload(accept='.pdf', multiple=False)
        self.classify_button = widgets.Button(description='Classify Test Document')
        self.classify_button.on_click(self.on_classify_click)

        self.new_topic_text = widgets.Text(description='Topic Name:', placeholder='Enter new topic name')
        self.new_topic_upload = widgets.FileUpload(accept='.pdf', multiple=True)
        self.add_topic_button = widgets.Button(description='Add New Topic')
        self.add_topic_button.on_click(self.on_add_topic_click)

        self.remove_topic_text = widgets.Text(description='Remove Topic:', placeholder='Enter topic name to remove')
        self.remove_topic_button = widgets.Button(description='Remove Topic')
        self.remove_topic_button.on_click(self.on_remove_topic_click)

        self.handle_feedback_button = widgets.Button(description='Process Feedback and Retrain')
        self.handle_feedback_button.on_click(self.on_handle_feedback_click)

        display(widgets.VBox([
            widgets.Label("Upload Test Document for Classification"),
            self.test_upload, self.classify_button,
            widgets.Label("Add New Topic (with Training Document)"),
            self.new_topic_text, self.new_topic_upload, self.add_topic_button,
            widgets.Label("Remove an Existing Topic"),
            self.remove_topic_text, self.remove_topic_button,
            self.handle_feedback_button

        ]))
        self.test_upload.observe(self.on_test_upload_change, names='value')

    def on_test_upload_change(self, change):
        """Process test document upload and save metadata for later classification."""
        if self.test_upload.value:
            for file_info in self.test_upload.value:
                filename = file_info['name'].replace(".pdf", "")
                file_content = file_info['content'].tobytes()
                # Process the PDF and save the cleaned text
                output_path, _ = self.processor.preprocess_pdf(file_content, filename)
                # Save metadata to be used by the classification function
                metadata = {"filename": filename, "file_path": output_path}
                with open("processed_file.pkl", "wb") as f:
                    pickle.dump(metadata, f)
                print("Test document processed and metadata saved.")

    def on_add_topic_click(self, b):
        """Handle the 'Add New Topic' button click event."""
        if self.new_topic_upload.value and self.new_topic_text.value:
            topic_name = self.new_topic_text.value.strip()
            for file_info in self.new_topic_upload.value:
                filename = file_info['name'].replace(".pdf", "")
                file_content = file_info['content'].tobytes()
                self.model_manager.add_new_topic(topic_name, file_content, filename, self.processor)
            # Reset input widgets
            self.new_topic_text.value = ""
            # self.new_topic_upload.value.clear()
            self.new_topic_upload._counter = 0  # Reset upload widget state
        else:
            print("Please provide both a topic name and a PDF document.")

    def on_remove_topic_click(self, b):
        """Handle the 'Remove Topic' button click event."""
        topic_name = self.remove_topic_text.value.strip()
        if topic_name:
            self.model_manager.remove_topic(topic_name)
            self.remove_topic_text.value = ""
        else:
            print("Please provide a topic name to remove.")


    def on_classify_click(self, b):
        """Classify the uploaded document and display results."""
        print("Classifying document...")
        try:
            metadata = pickle.load(open("processed_file.pkl", "rb"))
            filename, predicted_topic, confidence, explanation, source = self.model_manager.classify_document()
            result = {
                "Filename": filename,
                "Predicted Topic": predicted_topic,
                "Confidence": confidence,
                "Explanation": explanation,
                "Source": source
            }
            print("Classification Result:", json.dumps(result, indent=4))
        except Exception as e:
            print("Error during classification:", e)

    def on_handle_feedback_click(self, b):
        """Handle click to process user feedback and retrain the model."""
        print("Processing user feedback...")
        self.model_manager.handle_user_feedback()
# Initialize and display the interactive document classifier application
app = DocumentClassifierApp()


# Code with add/remove topic module, feedback module, explainability for ML method & Confidence Score for LLM

In [7]:
import os
import io
import pickle
import json
import re
import random
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from pypdf import PdfReader
import joblib
import boto3
from dotenv import load_dotenv
from botocore.config import Config
from nltk.corpus import stopwords

class DocumentProcessor:
    """
    Utility class for cleaning and preprocessing PDF documents.
    """
    @staticmethod
    def clean_text(text: str) -> str:
        # Remove non-ASCII characters, URLs, non-letter characters,
        # and perform lowercasing and whitespace normalization.
        text = re.sub(r'[^\x00-\x7F]+', ' ', text)
        text = re.sub(r'http\S+', '', text)
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        text = text.lower()
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    @staticmethod
    def remove_stop_words(text: str) -> str:
        stop_words = set(stopwords.words('english'))
        words = text.split()
        return ' '.join(word for word in words if word not in stop_words)

    def preprocess_pdf(self, file_content: bytes, filename: str) -> tuple[str, str]:
        try:
            pdf_stream = io.BytesIO(file_content)
            reader = PdfReader(pdf_stream)
            extracted_text = ""
            for page in reader.pages:
                text = page.extract_text()
                if text:
                    extracted_text += text + "\n"
            cleaned_text = self.clean_text(extracted_text)
            final_text = self.remove_stop_words(cleaned_text)
            output_path = os.path.join(os.getcwd(), f"{filename}_extracted.txt")
            with open(output_path, "w", encoding="utf-8") as text_file:
                text_file.write(final_text)
            return output_path, final_text
        except Exception as e:
            print(f"Error processing PDF: {e}")
            return None, None

class ModelManager:
    """
    Manages loading of the pre-trained models, classifies documents, and provides explainability.
    """
    def __init__(self, mapping_file_path: str = 'final_file_topic_mapping_v6.csv'):
        self.vectorizer_path = "tfidf_vectorizer.pkl"
        self.model_path = "rf_model.pkl"
        self.mapping_file_path = mapping_file_path
        self.load_models()
        self.update_unique_topics()
        self.processor = DocumentProcessor()

    def load_models(self) -> None:
        try:
            self.tfidf_vectorizer = joblib.load(self.vectorizer_path)
            self.rf_model = joblib.load(self.model_path)
            print("Models loaded successfully.")
        except Exception as e:
            print("Error loading models. Make sure the model files exist.", e)
            self.tfidf_vectorizer, self.rf_model = None, None

    def load_mapping_file(self) -> pd.DataFrame:
        if os.path.exists(self.mapping_file_path):
            return pd.read_csv(self.mapping_file_path)
        else:
            return pd.DataFrame(columns=["folder_name", "file_name"])

    def update_unique_topics(self) -> None:
        df = self.load_mapping_file()
        df = df.dropna(subset=["folder_name"])
        self.unique_topics = df['folder_name'].unique().tolist()
        self.unique_topics_str = ', '.join(self.unique_topics)
        print("Loaded topics:", self.unique_topics_str)

    def extract_intro_middle_conclusion(self, text: str, max_tokens: int = 20000) -> str:
        words = text.split()
        total_words = len(words)
        if total_words < 5000:
            ratios = (0.15, 0.15, 0.15)
        elif total_words < 20000:
            ratios = (0.08, 0.12, 0.08)
        elif total_words < 50000:
            ratios = (0.04, 0.08, 0.04)
        else:
            ratios = (0.02, 0.06, 0.02)
        intro_end = max(int(total_words * ratios[0]), 100)
        conclusion_start = max(int(total_words * (1 - ratios[2])), total_words - 100)
        middle = words[intro_end:conclusion_start]
        middle_sample_size = int(len(middle) * ratios[1])
        middle_sample = random.sample(middle, min(middle_sample_size, len(middle)))
        hybrid_text_words = words[:intro_end] + middle_sample + words[conclusion_start:]
        estimated_tokens = len(hybrid_text_words) * 1.3
        if estimated_tokens > max_tokens:
            allowed_words = int(max_tokens / 1.3)
            hybrid_text_words = hybrid_text_words[:allowed_words]
        return " ".join(hybrid_text_words)
    
    def rf_classify_document(self, text: str, confidence_threshold: float = 0.99) -> tuple[str, float]:
        text_tfidf = self.tfidf_vectorizer.transform([text])
        y_pred_proba = self.rf_model.predict_proba(text_tfidf)[0]
        max_prob = max(y_pred_proba)
        predicted_topic = self.rf_model.classes_[y_pred_proba.argmax()]
        if max_prob < confidence_threshold:
            return None, max_prob
        return predicted_topic, max_prob

    def evaluate_topic_with_llama(self, text: str) -> tuple[str, str]:
        try:
            prompt = f"""
            Analyze the following document sample and classify it into only one of these topics: {self.unique_topics_str}.
            After explaining your reasoning, clearly state the final topic and the confidence score at the end.
            
            Document:
            {text}
            
            Explanation: <Your explanation>
            Final Topic: <One of the topics from the list>
            Confidence Score: <0 to 100%>
            """
            formatted_prompt = f"""
            <|begin_of_text|>
            <|start_header_id|>user<|end_header_id|>
            {prompt}
            <|eot_id|>
            <|start_header_id|>assistant<|end_header_id|>
            """
            load_dotenv("codes.env")
            aws_access_key = os.environ.get("AWS_ACCESS_KEY_ID")
            aws_secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY")
            aws_region = os.environ.get("AWS_REGION")
            MODEL_ID_LLAMA = "us.meta.llama3-3-70b-instruct-v1:0"
            config = Config(read_timeout=1000)
            bedrock_client = boto3.client(
                "bedrock-runtime",
                region_name=aws_region,
                aws_access_key_id=aws_access_key,
                aws_secret_access_key=aws_secret_key,
                config=config
            )
            response = bedrock_client.invoke_model(
                modelId=MODEL_ID_LLAMA,
                body=json.dumps({
                    "prompt": formatted_prompt,
                    "max_gen_len": 512,
                    "temperature": 0,
                }),
                contentType="application/json"
            )
            response_body = json.loads(response['body'].read())
            response_text = response_body.get("generation", "").strip()
            match = re.search(r"Final Topic:\s*(.+)", response_text, re.IGNORECASE)
            predicted_topic = match.group(1).strip() if match else "Unknown"
            return predicted_topic, response_text
        except Exception as e:
            print(f"Error calling AWS Bedrock API: {e}")
            return "Unknown", "Error occurred during LLM call"

    def classify_document(self) -> tuple[str, str, float, str, str]:
        """
        Classify a document using the pre-processed text.
        First, the Random Forest classifier is used; if its confidence is low, the fallback LLM is invoked.
        """
        with open("processed_file.pkl", "rb") as f:
            metadata = pickle.load(f)
        file_path = metadata["file_path"]
        filename = metadata["filename"]

        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()

        sampled_text = self.extract_intro_middle_conclusion(content)
        predicted_topic, confidence = self.rf_classify_document(sampled_text)

        if predicted_topic:
            print(f"Random Forest Classification: {predicted_topic} (Confidence: {confidence:.2f})")
            return filename, predicted_topic, confidence, "-", "Random Forest Classification"

        predicted_topic, explanation = self.evaluate_topic_with_llama(sampled_text)
        print(f"LLM Classification: {predicted_topic}")
        return filename, predicted_topic, "-", explanation, "LLM Classification"

    def explain_prediction(self, text: str, method: str = "local") -> None:
        """
        Provide an explanation for a given prediction using one of several methods:
        
        - "local": A permutation-based local explanation that approximates feature importance.
        - "global": Uses the model's built-in global feature importances.
        - "shap": Computes and prints SHAP values for the input text (without visualization).
        
        Parameters:
            text (str): The document text to explain.
            method (str): The explanation method ("local", "global", or "shap").
        """
        import numpy as np
        # Convert the input text into a TF-IDF vector (dense format)
        text_tfidf = self.tfidf_vectorizer.transform([text])
        dense_vector = text_tfidf.toarray()  # shape: (1, n_features)
        feature_names = self.tfidf_vectorizer.get_feature_names_out()
        if len(feature_names) != dense_vector.shape[1]:
            feature_names = [f"f{i}" for i in range(dense_vector.shape[1])]
        
        # Get baseline prediction probability and predicted class.
        proba = self.rf_model.predict_proba(dense_vector)[0]
        predicted_class = self.rf_model.predict(dense_vector)[0]
        class_index = list(self.rf_model.classes_).index(predicted_class)
        baseline_prob = proba[class_index]
        
        if method == "local":
            # [Local explanation code unchanged...]
            candidate_indices = [i for i, val in enumerate(dense_vector[0]) if val > 0]
            candidate_indices = sorted(candidate_indices, key=lambda i: dense_vector[0][i], reverse=True)
            candidate_indices = candidate_indices[:20]
            
            feature_contributions = []
            for i in candidate_indices:
                modified_vector = dense_vector.copy()
                modified_vector[0][i] = 0.0  # Zero-out the feature's contribution
                new_prob = self.rf_model.predict_proba(modified_vector)[0][class_index]
                contribution = baseline_prob - new_prob
                feature_contributions.append((feature_names[i], contribution, dense_vector[0][i]))
            
            top_features = sorted(feature_contributions, key=lambda x: abs(x[1]), reverse=True)[:10]
            explanation_lines = [
                f"Predicted class: {predicted_class}",
                f"Baseline predicted probability: {baseline_prob:.4f}",
                "Top contributing features (feature: TF-IDF weight):"
            ]
            for feat, contrib, weight in top_features:
                explanation_lines.append(f"- {feat}: {weight:.4f}")
            print("\n".join(explanation_lines))
        
        elif method == "shap":
            # SHAP explanation: compute and print SHAP values per word.
            import shap
            # Compute SHAP values for the text
            explainer = shap.Explainer(self.rf_model.predict_proba, dense_vector, feature_names=feature_names)
            shap_values = explainer(dense_vector)
            # For multi-class, select SHAP values for the predicted class.
            shap_values_for_class = shap_values.values[0, :, class_index]
            
            # Only output words that are actually present (non-zero TF-IDF weight)
            nonzero_indices = [i for i, val in enumerate(dense_vector[0]) if val > 0]
            result_lines = [
                f"Word: {feature_names[i]}, TF-IDF: {dense_vector[0][i]:.4f}, SHAP: {shap_values_for_class[i]:.4f}"
                for i in nonzero_indices
            ]
            print("SHAP values for respective words:")
            print("\n".join(result_lines))
        
        else:
            print(f"Explainability method '{method}' not implemented.")

class DocumentClassifierApp:
    """
    Interactive application for document classification and explanation.
    It allows the user to upload a PDF file, classify it, and then view explanation details.
    """
    def __init__(self):
        self.processor = DocumentProcessor()
        self.model_manager = ModelManager()
        self.setup_widgets()

    def setup_widgets(self):
        # File upload widget for classification.
        self.test_upload = widgets.FileUpload(accept='.pdf', multiple=False)
        self.classify_button = widgets.Button(description='Classify Test Document')
        self.classify_button.on_click(self.on_classify_click)

        # Button to display explanation for the classified document using global feature importance.
        self.explain_button = widgets.Button(description='Explain Prediction (Global)')
        self.explain_button.on_click(self.on_explain_click)

        # New button to display SHAP-based explanation.
        self.shap_explain_button = widgets.Button(description='Explain with SHAP')
        self.shap_explain_button.on_click(self.on_shap_explain_click)

        display(widgets.VBox([
            widgets.Label("Upload Test Document for Classification"),
            self.test_upload,
            self.classify_button,
            self.explain_button,
            self.shap_explain_button
        ]))
        self.test_upload.observe(self.on_test_upload_change, names='value')

    def on_test_upload_change(self, change):
        if self.test_upload.value:
            for file_info in self.test_upload.value:
                filename = file_info['name'].replace(".pdf", "")
                file_content = file_info['content'].tobytes()
                output_path, _ = self.processor.preprocess_pdf(file_content, filename)
                metadata = {"filename": filename, "file_path": output_path}
                with open("processed_file.pkl", "wb") as f:
                    pickle.dump(metadata, f)
                print("Test document processed and metadata saved.")

    def on_classify_click(self, b):
        print("Classifying document...")
        try:
            filename, predicted_topic, confidence, explanation, source = self.model_manager.classify_document()
            result = {
                "Filename": filename,
                "Predicted Topic": predicted_topic,
                "Confidence": confidence,
                "Explanation": explanation,
                "Source": source
            }
            print("Classification Result:", json.dumps(result, indent=4))
        except Exception as e:
            print("Error during classification:", e)

    def on_explain_click(self, b):
        """
        After classification, extract the document text and show the explanation using global feature importances.
        """
        try:
            with open("processed_file.pkl", "rb") as f:
                metadata = pickle.load(f)
            file_path = metadata["file_path"]
            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()
            sampled_text = self.model_manager.extract_intro_middle_conclusion(content)
            self.model_manager.explain_prediction(sampled_text, method="local")
        except Exception as e:
            print("Error during explanation:", e)

    def on_shap_explain_click(self, b):
        """
        After classification, extract the document text and print SHAP values for explanation.
        """
        try:
            with open("processed_file.pkl", "rb") as f:
                metadata = pickle.load(f)
            file_path = metadata["file_path"]
            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()
            sampled_text = self.model_manager.extract_intro_middle_conclusion(content)
            self.model_manager.explain_prediction(sampled_text, method="shap")
        except Exception as e:
            print("Error during SHAP explanation:", e)

# Initialize and display the interactive document classifier application.
app = DocumentClassifierApp()

Models loaded successfully.
Loaded topics: Marketing and Public Communication, Annual Reports, Investment and Market Research, Taxation, Risk Management, Audit Reports, Consumer Finance, Financial Regulations, Anti Money Laundering, Technology, Loans, Mergers and Acquisitions, Derivatives, Employment, Non-Disclosure Agreements (NDA), Partnerships, Client Agreements, Strategic, Administrative, Operational
